# Message Sensation Value (MSV) Computational Model Development

This notebook documents the process used to develop the MSV computational model.

This notebook supplements the manuscript: **A Computational Model of Message Sensation Value in Short Video Multimodal Features that Predicts Sensory and Behavioral Engagement** by Haoning Xue, Jingwen Zhang, Xiaohui Wang, Diane Dagyong Kim, and Yunya Song.

If you have any questions, feel free to contact **Haoning Xue** via haoning.xue@utah.edu.

In [1]:
import pandas as pd
import numpy as np
import datetime
import re, csv
import math, statistics
import os, sys, glob, io
import pickle
import json
import joblib

In [2]:
import statsmodels.api as sm
from scipy.stats import spearmanr
import seaborn as sns
import scipy.stats as stats
import xgboost as xgb

In [3]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.tree import plot_tree

## Set-up

In [4]:
""" Version check """
import sys
import importlib.metadata as metadata

loaded_packages = {}

for module_name in sys.modules:
    top_level = module_name.split(".")[0]
    try:
        version = metadata.version(top_level)
        loaded_packages[top_level] = version
    except metadata.PackageNotFoundError:
        pass

for name, version in sorted(loaded_packages.items()):
    print(f"{name}=={version}")

IPython==8.26.0
appnope==0.1.4
asttokens==2.4.1
cffi==2.0.0
charset_normalizer==3.3.2
comm==0.2.2
cycler==0.12.1
datetime==5.5
debugpy==1.8.5
decorator==4.4.2
defusedxml==0.7.1
executing==2.0.1
ipykernel==6.29.5
ipywidgets==8.1.3
jedi==0.19.1
joblib==1.4.2
jupyter_client==8.6.2
jupyter_core==5.7.2
kiwisolver==1.4.5
matplotlib==3.9.2
numpy==2.0.1
packaging==24.1
pandas==2.2.2
parso==0.8.4
patsy==0.5.6
platformdirs==4.2.2
prompt_toolkit==3.0.47
psutil==6.0.0
pure_eval==0.2.3
pygments==2.18.0
pyparsing==3.1.2
pytz==2024.1
scipy==1.14.1
seaborn==0.13.2
six==1.16.0
stack_data==0.6.3
statsmodels==0.14.5
threadpoolctl==3.5.0
tornado==6.4.1
traitlets==5.14.3
wcwidth==0.2.13
xgboost==3.0.5


In [5]:
pd.set_option('display.float_format', '{:.10f}'.format)
np.set_printoptions(suppress=True, precision=6)

In [6]:
df = pd.read_csv('../data/msv_df_imputed.csv')

In [7]:
feature_11 = ['tempo', 'loudness', 'spec_centroid',
              'brightness', 'saturation', 'entropy', 'warmness',
              'facec_ps', 'shotc_ps', 'faced_ps', 'shotd_ps']

## Pre-processing

In [8]:
""" standardize features """

std_df = df.copy()
scaler = StandardScaler()
std_df[feature_11] = scaler.fit_transform(std_df[feature_11])

In [9]:
""" train, test split """
x = np.array(std_df[feature_11])
y = np.array(std_df['perceived_msv'])

X_train, X_test, y_train, y_test = train_test_split(std_df[feature_11], std_df['perceived_msv'], test_size=0.1, random_state=0)

train_df = pd.DataFrame(X_train)
train_df.columns = feature_11
train_df['y'] = y_train

test_df = pd.DataFrame(X_test)
test_df.columns = feature_11
test_df['y'] = y_test

In [10]:
np.var(y_test)

np.float64(0.663636597441327)

## Evaluation metrics

In [11]:
def get_eval(model, X_test, y_test):
    n = len(test_df)
    p = len(X_test.columns)

    if 'statsmodels' in str(type(model)):
        sm_df = pd.DataFrame(X_test)
        sm_df = sm.add_constant(sm_df)
        sm_df['predicted'] = model.predict(sm_df)
        sm_df['y'] = y_test
        model_mse = mean_squared_error(sm_df['predicted'], sm_df['y'])
        print("test corr: ", spearmanr(sm_df['predicted'], sm_df['y']))
    else:
        model_mse = mean_squared_error(model.predict(X_test), y_test)
        print("test corr: ", spearmanr(model.predict(X_test), y_test))

    RSS = model_mse * n  # Residual Sum of Squares
    AIC = 2*p + n*np.log(RSS/n)
    BIC = n*np.log(RSS/n) + p*np.log(n)

    print("baseline var:", np.var(y_test))
    print("mse:", model_mse)
    print("AIC:", AIC)
    print("BIC:", BIC)

## Model 1: linear regression

In [12]:
# data preprocessing
model_1_x = train_df[feature_11]
model_1_y = train_df['y']
model_1_x = sm.add_constant(model_1_x)

# model fitting
model_1 = sm.OLS(model_1_y, model_1_x).fit()

# model summary
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.147
Model:                            OLS   Adj. R-squared:                  0.138
Method:                 Least Squares   F-statistic:                     16.69
Date:                Tue, 28 Apr 2026   Prob (F-statistic):           1.11e-30
Time:                        12:07:32   Log-Likelihood:                -1223.7
No. Observations:                1080   AIC:                             2471.
Df Residuals:                    1068   BIC:                             2531.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             3.5039      0.023    152.320

In [13]:
get_eval(model_1, X_test, y_test)

test corr:  SignificanceResult(statistic=np.float64(0.4418782717566304), pvalue=np.float64(4.362566090291886e-07))
baseline var: 0.663636597441327
mse: 0.5583132617797736
AIC: -47.940208770609715
BIC: -17.277799600007214


## Model 2: polynomial regression

In [14]:
polynomial_features = PolynomialFeatures(degree=2, include_bias=False)
model_2 = Pipeline([("polynomial_features", polynomial_features), ("linear_regression", LinearRegression())])
model_2.fit(X_train, y_train)

Pipeline(steps=[('polynomial_features', PolynomialFeatures(include_bias=False)),
                ('linear_regression', LinearRegression())])

In [15]:
get_eval(model_2, X_test, y_test)

test corr:  SignificanceResult(statistic=np.float64(0.5441952930555524), pvalue=np.float64(1.3258248383089193e-10))
baseline var: 0.663636597441327
mse: 0.538323861237941
AIC: -52.315391284631204
BIC: -21.652982114028703


## Model 3: random forest regression

In [16]:
model_3 = RandomForestRegressor(random_state=0,
                                max_depth=7, max_features='log2', min_samples_split=5, n_estimators=30)
model_3.fit(X_train, y_train)

RandomForestRegressor(max_depth=7, max_features='log2', min_samples_split=5,
                      n_estimators=30, random_state=0)

In [17]:
get_eval(model_3, X_test, y_test)

test corr:  SignificanceResult(statistic=np.float64(0.6153258501330042), pvalue=np.float64(7.551916036759795e-14))
baseline var: 0.663636597441327
mse: 0.4481742326668733
AIC: -74.30878519652698
BIC: -43.64637602592448


In [18]:
model_3.score(X_test, y_test)

0.3246692023995904

In [19]:
# joblib.dump(model_3, '../model/msv_model.joblib')
# joblib.dump(scaler, '../model/msv_scaler.joblib')

## Model 4: xgboost regression

In [20]:
model_4 = xgb.XGBRegressor(objective ='reg:squarederror', random_state = 0,
                           colsample_bytree=0.5, learning_rate=0.1, max_depth=3, n_estimators=50, reg_alpha=0.1, reg_lambda=1.5)
model_4.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.5, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=50,
             n_jobs=None, num_parallel_tree=None, ...)

In [21]:
get_eval(model_4, X_test, y_test)

test corr:  SignificanceResult(statistic=np.float64(0.5785313088914167), pvalue=np.float64(4.507357877946859e-12))
baseline var: 0.663636597441327
mse: 0.4764373089130481
AIC: -66.97029563699199
BIC: -36.30788646638949


## Model 5: support vector regression

In [22]:
# Create the SVR model
model_5 = SVR(C=3, epsilon=0.2, gamma=0.01)

# Train the model
model_5.fit(X_train, y_train)

SVR(C=3, epsilon=0.2, gamma=0.01)

In [23]:
get_eval(model_5, X_test, y_test)

test corr:  SignificanceResult(statistic=np.float64(0.5079563507757102), pvalue=np.float64(3.180884032727699e-09))
baseline var: 0.663636597441327
mse: 0.49137324798811727
AIC: -63.26615129307348
BIC: -32.603742122470976
